In [1]:
import os
import pandas as pd
import pulp

In [2]:
os.chdir('boarding_group_assignment')
!ls

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
poc.ipynb  resource


In [3]:
df_students = pd.read_csv('resource/students.csv')
df_cars     = pd.read_csv('resource/cars.csv')

In [4]:
prob = pulp.LpProblem('ClubCarProblem', pulp.LpMinimize)  # Generate Instance.

In [6]:
# List
students = df_students['student_id'].tolist()  # 学生 List
cars = df_cars['car_id'].tolist()  # 車 List
grades = list(range(1, 5, 1))
students_cars = [(student, car) for student in students for car in cars]  # 学生の車の Pair の List
licensers = df_students.loc[df_students['license'] == 1, 'student_id']  # 免許を持っている学生の List
students_grades = {grade: df_students.loc[df_students['grade'] == grade, 'student_id'] for grade in
                   grades}  # 学年が grade の学生 List
students_male = df_students.loc[df_students['gender'] == 0, 'student_id']
students_female = df_students.loc[df_students['gender'] == 1, 'student_id']

In [7]:
# 定数
# 車の乗車定員の定数
CAR_CAPACITY = df_cars['capacity'].tolist()

In [8]:
# 変数
# 学生をどの車に割り当てるかを変数として定義
x = pulp.LpVariable.dicts('x', students_cars, cat='Binary')

In [13]:
# 制約
# 1. 各学生を１つの車に割り当てる
for student in students:
    prob += pulp.lpSum([x[student, car] for car in cars]) == 1

# 2. 法規則に関する制約: 各車には乗車定員より多く乗ることができない
for car in cars:
    prob += pulp.lpSum([x[student, car] for student in students]) <= CAR_CAPACITY[car]

# 3. 法規制に関する制約: 各車に Driver を１人以上割り当てる
for car in cars:
    prob += pulp.lpSum([x[licenser, car] for licenser in licensers]) >= 1

# 4. 懇親を目的とした制約: 各車に各学年の学生を１人以上割り当てる
for car in cars:
    for car in cars:
        for grade in grades:
            prob += pulp.lpSum([x[student_grade, car] for student_grade in students_grades[grade]]) >= 1

# 5. Gender Balance を考慮した制約: 各車に男性を１人以上割り当てる
for car in cars:
    prob += pulp.lpSum([x[student_male, car] for student_male in students_male]) >= 1

# 6. Gender Balance を考慮した制約: 各車に女性を１人以上割り当てる
for car in cars:
    prob += pulp.lpSum([x[student_female, car] for student_female in students_female]) >= 1

In [14]:
# 求解
status = prob.solve()

f'status: {pulp.LpStatus[status]}'

Welcome to the CBC MILP Solver 
Version: 2.9.0 
Build Date: Feb 12 2015 

command line - /home/peta/.local/share/virtualenvs/py_math_opt_2-18519I2X/lib/python3.8/site-packages/pulp/apis/../solverdir/cbc/linux/64/cbc /tmp/0ae51604f51e465d9e019916c04771a2-pulp.mps branch printingOptions all solution /tmp/0ae51604f51e465d9e019916c04771a2-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 245 COLUMNS
At line 2215 RHS
At line 2456 BOUNDS
At line 2602 ENDATA
Problem MODEL has 240 rows, 145 columns and 1680 elements
Coin0008I MODEL read with 0 errors
Continuous objective value is 0 - 0.00 seconds
Cgl0004I processed model has 72 rows, 144 columns (144 integer (144 of which binary)) and 672 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of 0
Cbc0038I Before mini branch and bound, 144 integers at bound fixed and 0 continuous
Cbc0038I Mini branch and bound did not improve solution (0.00 seconds)
Cbc0038I After 0.00 seconds 

'status: Optimal'

In [15]:
# 最適化結果の表示
# 各車に割り当てられている学生の List を Dict に格納（車ID → 割り当てられた学生の List）
car2students = {car: [student for student in students if x[student, car].value() == 1] for car in cars}

# 各車の乗車定員（車ID → 乗車定員）
max_people = dict(zip(df_cars['car_id'], df_cars['capacity']))

for car, students in car2students.items():
    print(f"車ID: {car}")
    print(f"学生数（乗車定員）: {len(students)}({max_people[car]})")
    print(f"学生: {students}\n")

車ID: 0
学生数（乗車定員）: 4(6)
学生: [0, 5, 18, 23]

車ID: 1
学生数（乗車定員）: 4(6)
学生: [2, 7, 16, 21]

車ID: 2
学生数（乗車定員）: 4(5)
学生: [9, 11, 12, 14]

車ID: 3
学生数（乗車定員）: 4(4)
学生: [3, 6, 17, 20]

車ID: 4
学生数（乗車定員）: 4(5)
学生: [1, 8, 10, 15]

車ID: 5
学生数（乗車定員）: 4(5)
学生: [4, 13, 19, 22]

